In [ ]:
import warnings
warnings.filterwarnings('ignore')

In [ ]:
import tensorflow as tf
from tensorflow import keras
import numpy as np
import time
import psutil
import os
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix
tf.config.experimental.list_physical_devices()

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow import keras
from keras.models import Sequential
from keras.layers import Dense, Dropout, Activation, Lambda, Bidirectional
from keras.layers import Embedding
from keras.layers import Convolution1D,MaxPooling1D, Flatten
from sklearn.model_selection import train_test_split
import pandas as pd

from sklearn.preprocessing import Normalizer
from keras.models import Sequential
from keras.layers import Convolution1D, Dense, Dropout, Flatten, MaxPooling1D
import h5py
from keras import callbacks
from keras.layers import LSTM, GRU, SimpleRNN
from keras.callbacks import CSVLogger
from keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau, CSVLogger
from sklearn.metrics import (precision_score, recall_score,f1_score, accuracy_score, confusion_matrix)
from sklearn import metrics
from tqdm import tqdm
import os
import polars as pl
import matplotlib.pyplot as plt
import seaborn as sns
from itertools import chain

from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import MinMaxScaler, StandardScaler
label_encoder = LabelEncoder()
scaler = StandardScaler()
colors = ['#900C3F', '#35155D', '#B0578D', '#618264', '#001524', '#BEADFA', '#272829', '#687EFF', '#004225', '#FFB000', 'red', 'blue', 'orange']

In [ ]:
X_columns = [
    'flow_duration', 'Header_Length', 'Protocol Type', 'Duration',
       'Rate', 'Srate', 'Drate', 'fin_flag_number', 'syn_flag_number',
       'rst_flag_number', 'psh_flag_number', 'ack_flag_number',
       'ece_flag_number', 'cwr_flag_number', 'ack_count',
       'syn_count', 'fin_count', 'urg_count', 'rst_count',
    'HTTP', 'HTTPS', 'DNS', 'Telnet', 'SMTP', 'SSH', 'IRC', 'TCP',
       'UDP', 'DHCP', 'ARP', 'ICMP', 'IPv', 'LLC', 'Tot sum', 'Min',
       'Max', 'AVG', 'Std', 'Tot size', 'IAT', 'Number', 'Magnitue',
       'Radius', 'Covariance', 'Variance', 'Weight',
]
y_column = 'label'

In [ ]:
df3 = pd.read_csv('/kaggle/input/ciciot2023-25f/25f-concatenated_file.csv')
df3.shape

In [ ]:
df3.head()

In [ ]:
class_counts = df3['label'].value_counts()
print(class_counts)

In [ ]:
# Dataset
X = df3[X_columns].values.astype(np.float32)
y = df3[y_column].values

In [ ]:
num_features = X.shape[1]
num_classes = len(np.unique(y))
print(num_features)
print(num_classes)

In [ ]:

# First, split the data into 80% training+validation and 20% testing
X_train_val, X_test, y_train_val, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Next, split the 80% training+validation data into 70% training and 10% validation
X_train, X_val, y_train, y_val = train_test_split(X_train_val, y_train_val, test_size=0.125, random_state=42, stratify=y_train_val)


In [ ]:
yy = label_encoder.fit(y)
y_name_mapping = dict(zip(yy.classes_, yy.transform(yy.classes_)))
print(y_name_mapping)

In [ ]:
# Feature scaling
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled   = scaler.transform(X_val)
X_test_scaled  = scaler.transform(X_test)

In [ ]:
# Reshape for RNN ((samples, time_steps = 1, features = F))
# X_train_scaled_rnn = X_train_scaled[..., np.newaxis]
# X_val_scaled_rnn   = X_val_scaled[..., np.newaxis]
# X_test_scaled_rnn  = X_test_scaled[..., np.newaxis]
X_train_scaled_rnn = X_train_scaled[:, np.newaxis, :]
X_val_scaled_rnn   = X_val_scaled[:, np.newaxis, :]
X_test_scaled_rnn  = X_test_scaled[:, np.newaxis, :]

print(X_train_scaled_rnn.shape)
print(X_val_scaled_rnn.shape)
print(X_test_scaled_rnn.shape)

In [ ]:
# Label encoding
label_encoder = LabelEncoder()
y_train_enc = label_encoder.fit_transform(y_train).astype(np.int32)
y_val_enc   = label_encoder.transform(y_val).astype(np.int32)
y_test_enc  = label_encoder.transform(y_test).astype(np.int32)

In [ ]:
print(X_train.shape)
print(X_val.shape)
X_test.shape 

In [ ]:
y_train

In [ ]:
X_train

In [ ]:
X_test

In [ ]:
print("Train label counts:\n", pd.Series(y_train_enc).value_counts())
print("Val label counts:\n", pd.Series(y_val_enc).value_counts())
print("Test label counts:\n", pd.Series(y_test_enc).value_counts())


In [ ]:
@tf.keras.utils.register_keras_serializable()
class LoggedLSTMCell(tf.keras.layers.Layer):

    def __init__(self, units, eps=1e-8, sparsity_threshold=0.1, **kwargs):
        super().__init__(**kwargs)
        self.units = units
        self.state_size = [units, units]
        self.output_size = units
        self.eps = eps
        self.sparsity_threshold = sparsity_threshold

        # ---- Gate stats ----
        self.i_entropy = tf.Variable(0.0, trainable=False)
        self.f_entropy = tf.Variable(0.0, trainable=False)
        self.o_entropy = tf.Variable(0.0, trainable=False)

        self.i_sparse = tf.Variable(0.0, trainable=False)
        self.f_sparse = tf.Variable(0.0, trainable=False)
        self.o_sparse = tf.Variable(0.0, trainable=False)

        self.step_count = tf.Variable(0.0, trainable=False)

    def build(self, input_shape):
        d = input_shape[-1]

        self.W = self.add_weight(shape=(d, 4*self.units),
                                 initializer="glorot_uniform")
        self.U = self.add_weight(shape=(self.units, 4*self.units),
                                 initializer="orthogonal")
        self.b = self.add_weight(shape=(4*self.units,),
                                 initializer="zeros")

    def call(self, inputs, states):
        h_prev, c_prev = states

        z = tf.matmul(inputs, self.W) + tf.matmul(h_prev, self.U) + self.b

        i, f, c_hat, o = tf.split(z, 4, axis=1)

        i_t = tf.nn.sigmoid(i)
        f_t = tf.nn.sigmoid(f)
        o_t = tf.nn.sigmoid(o)
        c_hat = tf.nn.tanh(c_hat)

        c_t = f_t * c_prev + i_t * c_hat
        h_t = o_t * tf.nn.tanh(c_t)

        # ---- Update stats ----
        self.step_count.assign_add(1.0)

        for gate, ent_var, sp_var in [
            (i_t, self.i_entropy, self.i_sparse),
            (f_t, self.f_entropy, self.f_sparse),
            (o_t, self.o_entropy, self.o_sparse),
        ]:
            # ---- Entropy ----
            p = gate / (tf.reduce_sum(gate, axis=0, keepdims=True) + self.eps)
            entropy = -tf.reduce_mean(
                tf.reduce_sum(p * tf.math.log(p + self.eps), axis=0)
            )
            ent_var.assign_add(entropy)

            # ---- Sparsity ----
            sparse = tf.reduce_mean(
                tf.cast(gate < self.sparsity_threshold, tf.float32)
            )
            sp_var.assign_add(sparse)

        return h_t, [h_t, c_t]

    def reset_gate_stats(self):
        self.i_entropy.assign(0.0)
        self.f_entropy.assign(0.0)
        self.o_entropy.assign(0.0)

        self.i_sparse.assign(0.0)
        self.f_sparse.assign(0.0)
        self.o_sparse.assign(0.0)

        self.step_count.assign(0.0)

    def get_gate_statistics(self):
        steps = tf.maximum(self.step_count, 1.0)

        return {
            "entropy": {
                "i": (self.i_entropy / steps).numpy(),
                "f": (self.f_entropy / steps).numpy(),
                "o": (self.o_entropy / steps).numpy(),
            },
            "sparsity": {
                "i": (self.i_sparse / steps).numpy(),
                "f": (self.f_sparse / steps).numpy(),
                "o": (self.o_sparse / steps).numpy(),
            }
        }


In [ ]:
def build_logged_lstm(num_features, num_classes):

    inputs = tf.keras.Input(shape=(1, num_features))

    x = tf.keras.layers.Conv1D(128, 3, padding="same", activation="relu")(inputs)
    x = tf.keras.layers.Conv1D(64, 3, padding="same", activation="relu")(x)

    cell = LoggedLSTMCell(units=70)
    rnn_layer = tf.keras.layers.RNN(cell)

    x = rnn_layer(x)

    outputs = tf.keras.layers.Dense(num_classes, activation="softmax")(x)

    model = tf.keras.Model(inputs, outputs)

    model.compile(
        optimizer="adam",
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )

    return model, rnn_layer


In [ ]:
model_lstm, rnn_lstm = build_logged_lstm(num_features, 34)

model_lstm.fit(
    X_train_scaled_rnn,
    y_train_enc,
    epochs=30,
    batch_size=1000,
    verbose=1
)

stats_lstm = rnn_lstm.cell.get_gate_statistics()

print("Entropy:", stats_lstm["entropy"])
print("Sparsity:", stats_lstm["sparsity"])


In [ ]:
model_lstm.summary()

In [ ]:
val_loss_before, val_acc_before = model_lstm.evaluate(X_val_scaled_rnn, y_val_enc, verbose=0)
test_loss_before, test_acc_before = model_lstm.evaluate(X_test_scaled_rnn, y_test_enc, verbose=0)
print(f"LSTM Val Accuracy : {val_acc_before:.4f}")
print(f"LSTM Test Accuracy: {test_acc_before:.4f}")

In [ ]:
y_pred_LSTM = np.argmax(cnn_lstm_model.predict(X_test_scaled_rnn, verbose=0), axis=-1)
Laccuracy = accuracy_score(y_test_enc, y_pred_LSTM)
Lprecision = precision_score(y_test_enc, y_pred_LSTM, average='weighted')
Lrecall = recall_score(y_test_enc,y_pred_LSTM,average='weighted')
Lf1 = f1_score(y_test_enc, y_pred_LSTM,average='weighted')

print(f"LSTM-Accuracy: {Laccuracy:.4f}")
print(f"LSTM-Precision: {Lprecision:.4f}")
print(f"LSTM-Recall: {Lrecall:.4f}")
print(f"LSTM-F1 Score: {Lf1:.4f}")

In [ ]:
!mkdir -p saved_results

In [ ]:
cnn_lstm_model.save(f"saved_results/LSTM-25F-34C.keras")